# Teacher Solution Notebook: Python + Excel darba uzdevumi (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/blob/main/notebooks/teacher_solution_latvia_excel_datasets_colab.ipynb)

Šī piezīmju grāmata ir pasniedzēja risinājumu versija kursa 5. tēmai **"Excel dokumentu apstrāde ar Python"** no 8 lekciju kursa struktūras.

Šī ir **Google Colab draudzīga** versija: ja mape `latvia_gov_excel_datasets` nav pieejama lokāli, piezīmju grāmata automātiski lejupielādēs publiskos Excel failus no repozitorija `ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python`.

Tā aptver šādus sintētisko datu komplektus:

1. `1_budget_reports` — vairāku Excel failu apvienošana  
2. `2_procurement_data` — nekārtīgu iepirkumu datu tīrīšana  
3. `3_employee_registry` — filtrēšana, grupēšana, atskaites  
4. `4_municipality_data` — Excel failu savienošana (merge/join)  
5. `5_mixed_quality_data` — haotiska reālas dzīves eksporta faila tīrīšana  
6. `6_reporting_template` — rezultātu ievietošana šablonā  
7. `7_chart_data` — diagrammas ievietošana ar `openpyxl`

## Pedagoģiskā pieeja

Šī nav tikai “pareizo atbilžu” kopa. Kods ir rakstīts tā, lai to varētu izmantot arī mācību demonstrācijām:
- vispirms ielasa datus,
- tad normalizē,
- tad apstrādā,
- tad eksportē jaunu Excel failu.

Dažās vietās redzamas arī alternatīvas pieejas, kuras var apspriest nodarbībā.

## 0. Vides sagatavošana

Šī versija paredzēta darbam gan lokāli, gan Google Colab vidē.

Datu avots:
- repozitorijs: `https://github.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python`
- datu mape: `https://github.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/tree/main/latvia_gov_excel_datasets`
- ģenerēšanas skripts: `scripts/create_excel_spreadsheets.py`

Nākamā koda šūna pārbaudīs bibliotēkas un, ja vajadzēs, lejupielādēs trūkstošos Excel failus no publiskā GitHub repozitorija.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
import re
from urllib.request import urlretrieve

IN_COLAB = "google.colab" in sys.modules

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages and IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
elif missing_packages:
    print("Missing local packages:", ", ".join(missing_packages))
    print("If you are not in Colab, install them manually before running the rest of the notebook.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.chart import BarChart, Reference

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


In [ ]:
REPO_OWNER = "ValRCS"
REPO_NAME = "RTU_Digitalas_Prasmes_Excel_VBA_Python"
REPO_BRANCH = "main"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}"
RAW_DATASET_BASE_URL = f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/{REPO_BRANCH}/latvia_gov_excel_datasets"

DATA_ROOT = Path("latvia_gov_excel_datasets")
OUTPUT_DIR = Path("teacher_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_FILES = [
    "README.txt",
    "1_budget_reports/budget_2025-01.xlsx",
    "1_budget_reports/budget_2025-02.xlsx",
    "1_budget_reports/budget_2025-03.xlsx",
    "1_budget_reports/budget_2025-04.xlsx",
    "1_budget_reports/budget_2025-05.xlsx",
    "1_budget_reports/budget_2025-06.xlsx",
    "2_procurement_data/procurement_raw.xlsx",
    "3_employee_registry/employees.xlsx",
    "4_municipality_data/population.xlsx",
    "4_municipality_data/budget.xlsx",
    "5_mixed_quality_data/mixed_data.xlsx",
    "6_reporting_template/report_template.xlsx",
    "7_chart_data/monthly_summary.xlsx",
]

def ensure_public_datasets(data_root: Path = DATA_ROOT):
    missing_files = [
        relative_path
        for relative_path in DATASET_FILES
        if not (data_root / relative_path).exists()
    ]

    for relative_path in missing_files:
        local_path = data_root / relative_path
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{RAW_DATASET_BASE_URL}/{relative_path}"
        urlretrieve(url, local_path)

    return missing_files

downloaded_files = ensure_public_datasets()

if downloaded_files:
    print(f"Downloaded {len(downloaded_files)} dataset files from {REPO_URL}.")
else:
    print("All dataset files are already available locally.")

print("Running in Colab:", IN_COLAB)
print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## Palīgfunkcijas

Šeit apkopotas atkārtoti izmantojamas funkcijas:
- datumu normalizēšana
- skaitļu pārvēršana no teksta
- piegādātāju nosaukumu tīrīšana
- kolonnu nosaukumu normalizēšana

In [ ]:
def parse_mixed_date(series: pd.Series) -> pd.Series:
    """Mēģina saprast vairākus datumu formātus."""
    return pd.to_datetime(series, errors="coerce", dayfirst=True)

def parse_amount(value):
    """Pārvērš dažādus EUR formātus uz float."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip()
    if not text:
        return np.nan

    # noņem atstarpes tūkstošu atdalīšanai
    text = text.replace(" ", "")

    # ja ir tikai komats, pieņemam to kā decimālatdalītāju
    if "," in text and "." not in text:
        text = text.replace(",", ".")
    # ja ir abi, pieņemam, ka komats ir tūkstošu vai lokālais formāts
    elif "," in text and "." in text:
        # mēģinām izņemt komatus kā tūkstošu atdalītājus
        text = text.replace(",", "")

    try:
        return float(text)
    except ValueError:
        return np.nan

def normalize_supplier_name(name: str) -> str:
    """Vienkāršota piegādātāju nosaukumu normalizācija."""
    if pd.isna(name):
        return name
    text = str(name).strip().lower()

    replacements = {
        '"': "",
        "sia ": "",
        "sia": "",
        "ltd.": "",
        "ltd": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    text = text.replace("ā", "a").replace("ē", "e").replace("ī", "i").replace("ū", "u").replace("ļ", "l").replace("ņ", "n").replace("ģ", "g").replace("ķ", "k").replace("š", "s").replace("ž", "z").replace("č", "c")
    text = re.sub(r"\s+", " ", text).strip()

    canonical_map = {
        "balttech risinajumi": "SIA BaltTech risinājumi",
        "baltech risinajumi": "SIA BaltTech risinājumi",
        "balttech": "SIA BaltTech risinājumi",
        "datu serviss": "SIA Datu Serviss",
        "ziemelu birojs": "SIA Ziemeļu Birojs",
        "e-parvalde": "SIA E-pārvalde",
        "e parvalde": "SIA E-pārvalde",
        "nova sistemas": "SIA Nova Sistēmas",
        "vieda vide": "SIA Viedā Vide",
        "latgales projekti": "SIA Latgales Projekti",
    }
    return canonical_map.get(text, str(name).strip())

def normalize_column_name(col: str) -> str:
    text = str(col).strip().lower()
    text = text.replace("department", "dept")
    text = text.replace("departments", "dept")
    text = text.replace("deparment", "dept")
    text = text.replace("municipality", "municipality")
    return text

# 1. Budget reports — vairāku Excel failu apvienošana

## Mērķis
- nolasīt visus mēneša budžeta failus no mapes,
- apvienot vienā tabulā,
- normalizēt datumus un summas,
- aprēķināt kopsavilkumu pa nodaļām un mēnešiem,
- eksportēt rezultātu uz jaunu Excel failu.

In [ ]:
budget_dir = DATA_ROOT / "1_budget_reports"
budget_files = sorted(budget_dir.glob("*.xlsx"))
budget_files

In [ ]:
budget_frames = []

for file in budget_files:
    df = pd.read_excel(file)
    df["Source File"] = file.name
    budget_frames.append(df)

budget_raw = pd.concat(budget_frames, ignore_index=True)
budget_raw.head()

In [ ]:
budget = budget_raw.copy()

budget["Amount EUR"] = budget["Amount EUR"].apply(parse_amount)
budget["Date Parsed"] = parse_mixed_date(budget["Date"])
budget["Month"] = budget["Date Parsed"].dt.to_period("M").astype("string")

# sakārtojam trūkstošās kategorijas
budget["Category"] = budget["Category"].fillna("Nav norādīts")

budget_clean = budget.dropna(subset=["Amount EUR", "Date Parsed"]).copy()

print("Raw rows:", len(budget_raw))
print("Clean rows:", len(budget_clean))
budget_clean.head()

In [ ]:
budget_summary_department = (
    budget_clean
    .groupby(["Month", "Department"], as_index=False)["Amount EUR"]
    .sum()
    .sort_values(["Month", "Amount EUR"], ascending=[True, False])
)

budget_summary_institution = (
    budget_clean
    .groupby(["Iestāde", "Department"], as_index=False)["Amount EUR"]
    .sum()
    .sort_values("Amount EUR", ascending=False)
)

budget_summary_department.head(12)

In [ ]:
budget_out = OUTPUT_DIR / "solution_budget_reports.xlsx"

with pd.ExcelWriter(budget_out, engine="openpyxl") as writer:
    budget_clean.to_excel(writer, sheet_name="CleanedData", index=False)
    budget_summary_department.to_excel(writer, sheet_name="ByDepartment", index=False)
    budget_summary_institution.to_excel(writer, sheet_name="ByInstitution", index=False)

budget_out

### Didaktiska piezīme

Šis uzdevums ir labs brīdis, lai parādītu atšķirību starp:
- manuālu failu atvēršanu Excel vidē,
- un reproducējamu failu apstrādi ar `for` ciklu un `pd.concat()`.

Galvenais konceptuālais ieguvums: **viens skripts aizstāj daudzus copy-paste soļus**.

# 2. Procurement data — nekārtīgu iepirkumu datu tīrīšana

## Mērķis
- nolasīt iepirkumu failu,
- normalizēt piegādātāju nosaukumus,
- sakārtot datumus un summas,
- iegūt kopsavilkumu pa piegādātājiem.

In [ ]:
proc_file = DATA_ROOT / "2_procurement_data" / "procurement_raw.xlsx"
proc_raw = pd.read_excel(proc_file)
proc_raw.head()

In [ ]:
proc = proc_raw.copy()

proc["Amount Clean"] = proc["Amount EUR"].apply(parse_amount)
proc["Date Parsed"] = parse_mixed_date(proc["Date"])
proc["Supplier Normalized"] = proc["Supplier Name"].apply(normalize_supplier_name)

proc_clean = proc.dropna(subset=["Amount Clean", "Date Parsed"]).copy()

supplier_summary = (
    proc_clean
    .groupby(["Supplier Normalized", "Contract Type"], as_index=False)
    .agg(
        Contracts=("Procedure No", "count"),
        Total_EUR=("Amount Clean", "sum"),
        Avg_EUR=("Amount Clean", "mean"),
    )
    .sort_values("Total_EUR", ascending=False)
)

supplier_summary.head(15)

In [ ]:
proc_out = OUTPUT_DIR / "solution_procurement.xlsx"

with pd.ExcelWriter(proc_out, engine="openpyxl") as writer:
    proc_clean.to_excel(writer, sheet_name="CleanedData", index=False)
    supplier_summary.to_excel(writer, sheet_name="SupplierSummary", index=False)

proc_out

### Ko te vērts uzsvērt auditorijai

Šeit ir ļoti tipiska situācija valsts pārvaldē:
- viens un tas pats piegādātājs dažādos failos vai ierakstos parādās dažādos rakstības veidos;
- Excel formulas vien palīdz tikai daļēji;
- Python ļauj ieviest skaidru un atkārtojamu normalizācijas loģiku.

# 3. Employee registry — filtrēšana, grupēšana un atskaite

## Mērķis
- atlasīt darbiniekus pēc kritērijiem,
- aprēķināt vidējās algas,
- sagatavot vienkāršu HR atskaiti.

In [ ]:
emp_file = DATA_ROOT / "3_employee_registry" / "employees.xlsx"
emp = pd.read_excel(emp_file)
emp["Start Date"] = parse_mixed_date(emp["Start Date"])
emp.head()

In [ ]:
high_salary = emp[emp["Salary"] > 2200].copy()

dept_summary = (
    emp.groupby("Department", as_index=False)
    .agg(
        Employees=("Employee ID", "count"),
        Avg_Salary=("Salary", "mean"),
        Max_Salary=("Salary", "max"),
        Min_Salary=("Salary", "min"),
    )
    .sort_values("Avg_Salary", ascending=False)
)

fte_summary = (
    emp.groupby("Department", as_index=False)["FTE"]
    .sum()
    .rename(columns={"FTE": "Total FTE"})
)

hr_report = dept_summary.merge(fte_summary, on="Department", how="left")
hr_report

In [ ]:
emp_out = OUTPUT_DIR / "solution_employee_registry.xlsx"

with pd.ExcelWriter(emp_out, engine="openpyxl") as writer:
    emp.to_excel(writer, sheet_name="AllEmployees", index=False)
    high_salary.to_excel(writer, sheet_name="HighSalary", index=False)
    hr_report.to_excel(writer, sheet_name="DepartmentReport", index=False)

emp_out

# 4. Municipality data — savienošana (merge/join)

## Mērķis
- ielasīt iedzīvotāju un budžeta failus,
- savienot tos pēc pašvaldības nosaukuma,
- aprēķināt budžetu uz vienu iedzīvotāju,
- atlasīt augstākos rādītājus.

In [ ]:
population_file = DATA_ROOT / "4_municipality_data" / "population.xlsx"
budget_file = DATA_ROOT / "4_municipality_data" / "budget.xlsx"

population = pd.read_excel(population_file)
budget_m = pd.read_excel(budget_file)

municipality = population.merge(budget_m, on=["Municipality", "Reference Year"], how="inner")
municipality["Budget per Capita"] = municipality["Budget"] / municipality["Population"]
municipality["Capital per Capita"] = municipality["Capital Expenditure"] / municipality["Population"]

municipality_sorted = municipality.sort_values("Budget per Capita", ascending=False)
municipality_sorted.head(10)

In [ ]:
mun_out = OUTPUT_DIR / "solution_municipality_analysis.xlsx"

with pd.ExcelWriter(mun_out, engine="openpyxl") as writer:
    municipality.to_excel(writer, sheet_name="MergedData", index=False)
    municipality_sorted.to_excel(writer, sheet_name="SortedByCapita", index=False)

mun_out

In [ ]:
# Vienkāršs vizuāls piemērs ar matplotlib
top10 = municipality_sorted.head(10)

plt.figure(figsize=(10, 5))
plt.bar(top10["Municipality"], top10["Budget per Capita"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("EUR uz 1 iedzīvotāju")
plt.title("Top 10 pašvaldības pēc budžeta uz vienu iedzīvotāju")
plt.tight_layout()
plt.show()

# 5. Mixed quality data — haotiska eksporta tīrīšana

Šis ir svarīgs uzdevums, jo tas ir tuvāks reālajai dzīvei nekā “skaistas tabulas”.

## Problēmas failā
- papildus virsraksta rindas,
- tukšas rindas,
- dažādi kolonnu virsraksti,
- skaitļi kā teksts,
- datumu formātu sajaukums,
- dublikāti.

In [ ]:
mixed_file = DATA_ROOT / "5_mixed_quality_data" / "mixed_data.xlsx"
mixed_raw = pd.read_excel(mixed_file, header=None)
mixed_raw.head(15)

In [ ]:
# Atrodam rindas, kur sākas faktiskie galvenes ieraksti
header_rows = mixed_raw.index[mixed_raw[0].astype(str).str.contains("Municipality", na=False)].tolist()
header_rows

In [ ]:
parts = []

for i, header_idx in enumerate(header_rows):
    start = header_idx + 1
    end = header_rows[i + 1] if i + 1 < len(header_rows) else len(mixed_raw)

    header = mixed_raw.iloc[header_idx].tolist()
    block = mixed_raw.iloc[start:end].copy()
    block.columns = header
    parts.append(block)

mixed = pd.concat(parts, ignore_index=True)
mixed.head()

In [ ]:
# Standartizējam kolonnas
mixed.columns = [str(c).strip() for c in mixed.columns]
rename_map = {}
for c in mixed.columns:
    low = c.lower()
    if low in ["dept", "department", "deparment", "department "]:
        rename_map[c] = "Department"
    elif low == "department":
        rename_map[c] = "Department"
    elif low == "municipality":
        rename_map[c] = "Municipality"
    elif low == "category":
        rename_map[c] = "Category"
    elif low == "amount":
        rename_map[c] = "Amount"
    elif low == "date":
        rename_map[c] = "Date"
    elif low == "approved":
        rename_map[c] = "Approved"

mixed = mixed.rename(columns=rename_map)

# izmetam pilnīgi tukšas rindas
mixed = mixed.dropna(how="all").copy()

# tīrām lauku vērtības
mixed["Amount Clean"] = mixed["Amount"].apply(parse_amount)
mixed["Date Parsed"] = parse_mixed_date(mixed["Date"])

if "Department" in mixed.columns:
    mixed["Department"] = (
        mixed["Department"]
        .astype(str)
        .str.replace("Dept:", "", regex=False)
        .str.replace("DEPARTMENT:", "", regex=False)
        .str.strip()
    )

if "Approved" in mixed.columns:
    approved_map = {
        "jā": True,
        "ja": True,
        "nē": False,
        "ne": False,
        "true": True,
        "false": False,
    }
    mixed["Approved Clean"] = mixed["Approved"].astype(str).str.strip().str.lower().map(approved_map)

mixed_clean = mixed.drop_duplicates().copy()
mixed_clean.head()

In [ ]:
mixed_summary = (
    mixed_clean
    .dropna(subset=["Amount Clean"])
    .groupby(["Municipality", "Department"], as_index=False)["Amount Clean"]
    .sum()
    .sort_values("Amount Clean", ascending=False)
)

mixed_out = OUTPUT_DIR / "solution_mixed_quality_data.xlsx"

with pd.ExcelWriter(mixed_out, engine="openpyxl") as writer:
    mixed_clean.to_excel(writer, sheet_name="CleanedData", index=False)
    mixed_summary.to_excel(writer, sheet_name="Summary", index=False)

mixed_out

### Mācību komentārs

Šī ir vieta, kur dalībnieki parasti saprot, ka:

- “Excel fails” nenozīmē “tīra tabula”;
- bieži vien vispirms ir jāatdala **dokumenta troksnis** no **datiem**;
- Python nav tikai aprēķinu rīks, bet arī **datu glābšanas rīks**.

# 6. Reporting template — rezultātu ievietošana šablonā

Mērķis: izmantot iepriekš aprēķināto kopsavilkumu un ierakstīt to sagatavotā Excel šablonā.

In [ ]:
template_file = DATA_ROOT / "6_reporting_template" / "report_template.xlsx"
report_output = OUTPUT_DIR / "solution_report_from_template.xlsx"

# izmantosim budžeta kopsavilkumu pēc nodaļām
report_data = (
    budget_clean.groupby("Department", as_index=False)["Amount EUR"]
    .sum()
    .sort_values("Amount EUR", ascending=False)
)

report_data

In [ ]:
# Vispirms saglabājam kopiju
import shutil
shutil.copy(template_file, report_output)

wb = load_workbook(report_output)
ws = wb["Kopsavilkums"]

ws["B3"] = "Demonstrācijas iestāde"
ws["B4"] = "2025-01 līdz 2025-06"

start_row = 7
for i, row in enumerate(report_data.itertuples(index=False), start=start_row):
    ws[f"A{i}"] = row.Department
    ws[f"B{i}"] = float(row[1])

# vienkāršs formatējums
for cell in ws[f"A6:B6"][0]:
    cell.font = Font(bold=True)
    cell.fill = PatternFill("solid", fgColor="D9EAF7")

wb.save(report_output)
report_output

# 7. Chart data — diagrammas ievietošana ar `openpyxl`

Šajā piemērā:
- nolasām kopsavilkuma failu,
- pārbaudām, vai diagramma jau ir,
- pievienojam vai pārrakstām diagrammu.

In [ ]:
chart_file = DATA_ROOT / "7_chart_data" / "monthly_summary.xlsx"
chart_output = OUTPUT_DIR / "solution_chart_data.xlsx"

import shutil
shutil.copy(chart_file, chart_output)

wb = load_workbook(chart_output)
ws = wb["Summary"]

# Pievienojam vēl vienu diagrammu zem esošās
chart = BarChart()
chart.title = "Izdevumi pa mēnešiem un nodaļām"
chart.y_axis.title = "EUR"
chart.x_axis.title = "Mēnesis"

data = Reference(ws, min_col=2, min_row=1, max_col=ws.max_column, max_row=ws.max_row)
cats = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)

chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)

ws.add_chart(chart, "J20")
wb.save(chart_output)

chart_output

# 8. Kopsavilkums pasniedzējam

## Ko šī piezīmju grāmata demonstrē

- failu apstrādi mapē (`glob`, `for`)
- vairāku Excel failu apvienošanu
- datumu un skaitļu normalizēšanu
- datu grupēšanu un summēšanu ar `pandas`
- datu savienošanu ar `merge`
- nekārtīgu failu strukturēšanu
- rezultātu eksportu uz jauniem Excel failiem
- `openpyxl` izmantošanu šabloniem un diagrammām

## Iespējamie diskusiju jautājumi nodarbībā

1. Kad pietiek ar `pandas`, un kad vajag `openpyxl`?
2. Kādus riskus rada manuāla datu kopēšana?
3. Kā nodrošināt, lai skripts būtu atkārtoti izmantojams nākamajā mēnesī?
4. Kā pārbaudīt, vai tīrīšanas noteikumi nav pārāk agresīvi?

## Nākamais solis

No šīs pasniedzēja versijas var viegli izveidot:
- vienkāršotu studentu notebook versiju,
- uzdevumu lapu bez pilna risinājuma,
- atsevišķu mājasdarba komplektu,
- vai Jupyter notebook ar tukšām šūnām, kur studenti paši pabeidz kodu.